# Causal Impact of Weekend Staffing Reallocation on Conversion Rate

**Domain:** Retail operations / HR planning — follow-up to [`staffing-conversion-rate-analysis`](https://github.com/marcomorabito94-mm/staffing-conversion-rate-analysis)  
**Method:** Difference-in-Differences (DiD) with Wasserstein-matched control group  
**Stack:** Python · pandas · numpy · seaborn · matplotlib · scipy · scikit-learn

---

## Context

A previous exploratory study identified a positive correlation between weekend staffing coverage and conversion rate (CR), with the strongest signal in the afternoon slot (15–17). **Forlì** — one of five stores flagged for high incremental potential — implemented a shift reallocation: redistributing existing weekend hours toward the 12–17 window without adding headcount.

## Objective

Measure whether the reallocation **causally** improved CR and revenue, isolating the intervention from seasonal trends via Difference-in-Differences against a matched control group.

## Key results

| Slot | ΔStaff DiD | ΔCR DiD | Revenue |
|---|---|---|---|
| Morning 8–11 | −1.17 | +0.64pp | €1,685 |
| Lunch 12–14 | +3.44 | +1.31pp | €2,424 |
| **Afternoon 15–17** | **+3.60** | **+2.94pp** | **€5,429** |
| Evening 18–close | −1.53 | +0.61pp | €966 |
| **TOTAL** | | **+1.37pp** | **€10,504** |

Zero additional headcount. 7 weekends (Nov 15 – Dec 31).


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wasserstein_distance, gaussian_kde
from sklearn.preprocessing import MinMaxScaler

pd.options.display.float_format = '{:.4f}'.format
sns.set_theme(style='whitegrid', palette='muted')

VARIANT_ID   = 47
VARIANT_NAME = 'Forlì'
CONTROL_IDS  = [16, 25, 29, 31, 46, 9, 11, 12, 30, 44]
slot_order   = ['morning_8-11', 'lunch_12-14', 'afternoon_15-17', 'late_afternoon_18-close']

## 1. Load data

In [ ]:
variant    = pd.read_csv('data/variant_store.csv')
all_stores = pd.read_csv('data/all_stores_did.csv')
background = pd.read_csv('data/all_stores_background.csv')

for df in [variant, all_stores, background]:
    df['date']            = pd.to_datetime(df['date'])
    df['cr']              = df['conversioni'] / df['ingressi'].replace(0, np.nan)
    df['staff_per_100']   = df['actual_staff'] / df['ingressi'].replace(0, np.nan) * 100
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df['planning_period'] = pd.Categorical(df['planning_period'],
        categories=['pre_planning', 'post_planning'], ordered=True)
    df['fasce_orarie']    = pd.Categorical(df['fasce_orarie'],
        categories=slot_order, ordered=True)

print(f"Variant:    {len(variant):,} rows")
print(f"DiD stores: {all_stores['store_id'].nunique()} stores, {len(all_stores):,} rows")
print(f"Background: {background['store_id'].nunique()} stores, {len(background):,} rows")
variant.head(3)

## 2. Did staffing actually change?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: staff distribution in 15-17 slot
filt = variant[variant['fasce_orarie'] == 'afternoon_15-17'].dropna(subset=['actual_staff'])
sns.boxplot(data=filt, x='planning_period', y='actual_staff',
            hue='period_type', showfliers=False, ax=axes[0])
axes[0].set_title('Staff on shift – slot 15–17\nPre vs Post, weekday vs weekend')
axes[0].set_xlabel('Planning period'); axes[0].set_ylabel('Staff on shift')

# Right: staff/100 entries by slot, weekends only
filt2 = variant[variant['period_type'] == 'weekend'].dropna(subset=['staff_per_100']).copy()
sns.boxplot(data=filt2, x='planning_period', y='staff_per_100',
            hue='fasce_orarie', showfliers=False, ax=axes[1])
axes[1].set_title('Staff / 100 entries by slot – weekends only\nPre vs Post')
axes[1].set_xlabel('Planning period'); axes[1].set_ylabel('Staff / 100 entries')

plt.tight_layout()
plt.savefig('outputs/01_staff_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** Afternoon 15–17 and lunch 12–14 show clear coverage increases on weekends post-intervention. Morning and evening decrease slightly — consistent with a pure reallocation of existing hours, not headcount addition.

## 3. Control group selection via Wasserstein distance

In [ ]:
variant_pre = all_stores[
    (all_stores['store_id'] == VARIANT_ID) &
    (all_stores['planning_period'] == 'pre_planning') &
    (all_stores['period_type'] == 'weekend')
].dropna(subset=['cr', 'staff_per_100', 'actual_staff'])

candidates_pre = all_stores[
    (all_stores['store_id'] != VARIANT_ID) &
    (all_stores['planning_period'] == 'pre_planning') &
    (all_stores['period_type'] == 'weekend')
].dropna(subset=['cr', 'staff_per_100', 'actual_staff'])

cols = ['cr', 'staff_per_100', 'actual_staff']
combined    = pd.concat([variant_pre[cols], candidates_pre[cols]], ignore_index=True)
scaler      = MinMaxScaler()
scaled_vals = pd.DataFrame(scaler.fit_transform(combined), columns=cols)
scaled_vals['store_id'] = pd.concat(
    [variant_pre[['store_id']], candidates_pre[['store_id']]], ignore_index=True
)['store_id'].values

v_scaled = scaled_vals[scaled_vals['store_id'] == VARIANT_ID]
distances = []
for sid in candidates_pre['store_id'].unique():
    c_scaled = scaled_vals[scaled_vals['store_id'] == sid]
    if len(c_scaled) < 5: continue
    d = (wasserstein_distance(v_scaled['cr'], c_scaled['cr']) +
         wasserstein_distance(v_scaled['staff_per_100'], c_scaled['staff_per_100']) +
         wasserstein_distance(v_scaled['actual_staff'], c_scaled['actual_staff']))
    distances.append({'store_id': sid, 'distance': round(d, 4), 'n_obs': len(c_scaled)})

dist_df = pd.DataFrame(distances).sort_values('distance').reset_index(drop=True)
print("Top 10 closest stores (selected as control group):")
dist_df.head(10)

In [ ]:
# Validate: KDE comparison using all 47 stores as background
ctrl_pre = background[
    (background['store_id'].isin(CONTROL_IDS)) &
    (background['planning_period'] == 'pre_planning') &
    (background['period_type'] == 'weekend')
].dropna(subset=['cr', 'staff_per_100', 'actual_staff'])

all_pre = background[
    (background['planning_period'] == 'pre_planning') &
    (background['period_type'] == 'weekend')
].dropna(subset=['cr', 'staff_per_100', 'actual_staff'])

v_pre = background[
    (background['store_id'] == VARIANT_ID) &
    (background['planning_period'] == 'pre_planning') &
    (background['period_type'] == 'weekend')
].dropna(subset=['cr', 'staff_per_100', 'actual_staff'])

metrics = {
    'actual_staff':  ('sum',    'Total hours worked (pre)'),
    'staff_per_100': ('mean',   'Staff coverage / 100 entries (pre)'),
    'cr':            ('median', 'Median CR (pre)')
}

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, (col, (fn, title)) in zip(axes, metrics.items()):
    c_vals = ctrl_pre.groupby('store_id')[col].agg(fn).values
    a_vals = all_pre.groupby('store_id')[col].agg(fn).values
    v_val  = v_pre[col].agg(fn)
    pad = (a_vals.max() - a_vals.min()) * 0.15
    x = np.linspace(a_vals.min() - pad, a_vals.max() + pad, 300)
    ax.plot(x, gaussian_kde(a_vals)(x), color='#F56403', label='All stores (47)', lw=2)
    ax.plot(x, gaussian_kde(c_vals)(x), color='#1E4D83', label='Control (10 stores)', lw=2)
    ax.axvline(v_val, color='red', ls='--', lw=2, label=VARIANT_NAME)
    ax.set_title(title); ax.set_yticks([]); ax.legend(fontsize=8)

plt.suptitle('Pre-period similarity: Control group vs all 47 stores vs Forlì',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('outputs/02_control_selection.png', dpi=150, bbox_inches='tight')
plt.show()

**Control group validity:**
- Forlì sits at the **50th percentile of the control group CR distribution** — well centered, not at the edge
- The control KDE (blue) is tighter than the full network (orange) and centered exactly on Forlì's values
- Hours worked and staff coverage distributions are nearly identical between Forlì and controls — validating the parallel trends assumption

## 4. Difference-in-Differences

**Design:**
- **Variant Δ** = post_CR − pre_CR for Forlì
- **Control Δ** = post_CR − pre_CR averaged across 10 matched stores
- **DiD** = Variant Δ − Control Δ → removes any common seasonal trend

Control stores show near-zero CR change (−0.14pp to +0.16pp), confirming the assumption holds.

In [ ]:
df = all_stores[all_stores['store_id'].isin([VARIANT_ID] + CONTROL_IDS)].copy()
df = df[(df['period_type'] == 'weekend') & (df['ingressi'] > 5)]
df = df.dropna(subset=['cr', 'staff_per_100'])
df['store_type'] = np.where(df['store_id'] == VARIANT_ID, 'variant', 'control')

agg = df.groupby(['fasce_orarie', 'store_type', 'planning_period']).agg(
    cr_median   = ('cr', 'median'),
    op_pressure = ('staff_per_100', 'mean'),
    n_obs       = ('cr', 'count')
).reset_index()

pivot = agg.pivot_table(
    index=['fasce_orarie', 'store_type'], columns='planning_period',
    values=['cr_median', 'op_pressure', 'n_obs']
).reset_index()
pivot.columns = ['_'.join(filter(None, map(str, c))).strip('_') for c in pivot.columns]
pivot['cr_diff']    = pivot['cr_median_post_planning'] - pivot['cr_median_pre_planning']
pivot['staff_diff'] = pivot['op_pressure_post_planning'] - pivot['op_pressure_pre_planning']

var_p  = pivot[pivot['store_type'] == 'variant'].copy()
ctrl_p = pivot[pivot['store_type'] == 'control'].copy()

final_table = var_p.merge(ctrl_p, on='fasce_orarie', suffixes=('_variant', '_control'))
final_table['cr_did']    = final_table['cr_diff_variant'] - final_table['cr_diff_control']
final_table['staff_did'] = final_table['staff_diff_variant'] - final_table['staff_diff_control']
final_table['fasce_orarie'] = pd.Categorical(
    final_table['fasce_orarie'], categories=slot_order, ordered=True)
final_table = final_table.sort_values('fasce_orarie').reset_index(drop=True)

summary = final_table[[
    'fasce_orarie', 'staff_diff_variant', 'cr_diff_variant',
    'staff_diff_control', 'cr_diff_control', 'staff_did', 'cr_did'
]].copy()
summary.columns = ['Slot','ΔStaff Variant','ΔCR Variant',
                   'ΔStaff Control','ΔCR Control','ΔStaff DiD','ΔCR DiD']
print(summary.to_string(index=False))
print(f"\nTotal DiD CR: {final_table['cr_did'].mean()*100:.2f}pp")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, fascia in zip(axes.flatten(), slot_order):
    row = final_table[final_table['fasce_orarie'] == fascia].iloc[0]
    periods = ['Pre', 'Post']
    s_v  = [row['op_pressure_pre_planning_variant'],  row['op_pressure_post_planning_variant']]
    s_c  = [row['op_pressure_pre_planning_control'],   row['op_pressure_post_planning_control']]
    cr_v = [row['cr_median_pre_planning_variant'],     row['cr_median_post_planning_variant']]
    cr_c = [row['cr_median_pre_planning_control'],     row['cr_median_post_planning_control']]
    vc, cc = '#D62728', '#0072B2'

    ms = max((max(s_v+s_c)-min(s_v+s_c))*0.5, 1.2)
    mc = max((max(cr_v+cr_c)-min(cr_v+cr_c))*0.6, 0.020)

    ax.plot(periods, s_v, marker='o', lw=2.5, color=vc, label='Staff – Forlì')
    ax.plot(periods, s_c, marker='o', lw=2.5, color=cc, label='Staff – Control')
    ax.set_ylabel('Staff / 100 entries', fontsize=9)
    ax.set_ylim(min(s_v+s_c)-ms, max(s_v+s_c)+ms)

    ax2 = ax.twinx()
    ax2.plot(periods, cr_v, marker='s', lw=2, ls='--', color=vc, alpha=0.85, label='CR – Forlì')
    ax2.plot(periods, cr_c, marker='s', lw=2, ls='--', color=cc, alpha=0.85, label='CR – Control')
    ax2.set_ylabel('CR (median)', fontsize=9)
    ax2.set_ylim(min(cr_v+cr_c)-mc, max(cr_v+cr_c)+mc)

    for x, y in zip(periods, s_v):
        ax.text(x, y+ms*0.3, f'{y:.2f}', color=vc, ha='center', fontsize=8, fontweight='bold')
    for x, y in zip(periods, s_c):
        ax.text(x, y-ms*0.35, f'{y:.2f}', color=cc, ha='center', fontsize=8, fontweight='bold')
    for x, y in zip(periods, cr_v):
        ax2.text(x, y+mc*0.35, f'{y:.1%}', color=vc, ha='center', fontsize=8)
    for x, y in zip(periods, cr_c):
        ax2.text(x, y-mc*0.40, f'{y:.1%}', color=cc, ha='center', fontsize=8)

    ax.set_title(
        f'{fascia}\nΔStaff (DiD): {row["staff_did"]:+.2f} | ΔCR (DiD): {row["cr_did"]*100:+.1f}pp',
        fontsize=9, fontweight='bold')
    l1, lb1 = ax.get_legend_handles_labels()
    l2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, lb1+lb2, fontsize=7, frameon=False, loc='upper left')

plt.suptitle(f'Staff coverage & CR – Weekend – {VARIANT_NAME} vs Control',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/03_did_by_slot.png', dpi=150, bbox_inches='tight')
plt.show()

**Findings — verified on synthetic data:**
- **Afternoon 15–17:** staffing DiD +3.60, CR DiD **+2.94pp** — primary target slot, strongest signal
- **Lunch 12–14:** staffing DiD +3.44, CR DiD **+1.31pp** — secondary effect from reallocation
- **Morning 8–11:** staffing DiD −1.17, CR DiD +0.64pp — modest positive despite coverage reduction
- **Evening 18–close:** staffing DiD −1.53, CR DiD +0.61pp — minimal impact as expected
- Control stores: CR drift near zero in all slots (−0.14pp to +0.16pp) ✓

## 5. Economic impact

In [ ]:
df_post_var = df[(df['store_type'] == 'variant') & (df['planning_period'] == 'post_planning')].copy()
df_post_var = df_post_var.merge(
    final_table[['fasce_orarie', 'cr_did']], on='fasce_orarie', how='left')
df_post_var['aov']        = df_post_var['cifra'] / df_post_var['conversioni'].replace(0, np.nan)
df_post_var['delta_conv'] = df_post_var['ingressi'] * df_post_var['cr_did']
df_post_var['delta_rev']  = df_post_var['delta_conv'] * df_post_var['aov']

econ = df_post_var.groupby('fasce_orarie').agg(
    delta_revenue = ('delta_rev',  'sum'),
    ingressi      = ('ingressi',   'sum'),
    cr_did_mean   = ('cr_did',     'mean'),
    aov_mean      = ('aov',        'mean')
).reset_index()
econ['rev_per_1ppt'] = econ['delta_revenue'] / (econ['cr_did_mean'] * 100)
econ['fasce_orarie'] = pd.Categorical(econ['fasce_orarie'], categories=slot_order, ordered=True)
econ = econ.sort_values('fasce_orarie').reset_index(drop=True)

# Total row (weighted average CR)
w_cr = (econ['cr_did_mean'] * econ['ingressi']).sum() / econ['ingressi'].sum()
total = pd.DataFrame([{
    'fasce_orarie': 'TOTAL',
    'delta_revenue': econ['delta_revenue'].sum(),
    'ingressi': econ['ingressi'].sum(),
    'cr_did_mean': w_cr,
    'aov_mean': econ['aov_mean'].mean(),
    'rev_per_1ppt': econ['delta_revenue'].sum() / (w_cr * 100)
}])
econ_full = pd.concat([econ, total], ignore_index=True)

econ_full['ΔCR DiD']                 = econ_full['cr_did_mean'].apply(lambda x: f'{x*100:.2f}pp')
econ_full['Incremental revenue (€)'] = econ_full['delta_revenue'].apply(lambda x: f'€{x:,.0f}')
econ_full['€ per +1pp CR']           = econ_full['rev_per_1ppt'].apply(lambda x: f'€{x:,.0f}')
econ_full['AOV (€)']                 = econ_full['aov_mean'].apply(lambda x: f'€{x:.0f}')

print(econ_full[['fasce_orarie','ΔCR DiD','ingressi','AOV (€)',
                  'Incremental revenue (€)','€ per +1pp CR']].to_string(index=False))
print(f"\nTotal incremental revenue: €{econ['delta_revenue'].sum():,.0f}")

**Economic result — verified numbers:**

The staffing reallocation generated **~€10.5k in incremental revenue** over 7 weekends at zero additional headcount cost.

- **Afternoon 15–17** contributes €5,429 (52% of total) with the highest revenue per pp of CR (€1,849/pp) — largest DiD effect and high-traffic slot
- **Lunch 12–14** contributes €2,424 — second-largest slot by CR impact
- Morning and evening together contribute €2,651 — residual effects from the reallocation

The result is proportional to Forlì's traffic volumes. The same redistribution logic applied to higher-traffic stores from the initial shortlist would generate proportionally larger returns.

---

## Conclusions

- Weekend shift redistribution at Forlì — **zero additional headcount** — produced a **+1.37pp CR improvement (DiD)**
- Effect concentrated in **afternoon 15–17** (+2.94pp) and **lunch 12–14** (+1.31pp), the two slots where coverage increased most
- Incremental revenue: **~€10.5k over 7 weekends**
- Staff coverage (staff/100 entries) confirmed as a **causal lever** for weekend performance, not just a correlation
- Model replicable to the other four stores in the preliminary shortlist, with higher expected returns

---

*Companion: [`staffing-conversion-rate-analysis`](../staffing-conversion-rate-analysis) — exploratory study identifying the staffing→CR correlation and store-level opportunity.*
